<div style="text-align: justify; max-width: 90ch">

# Lecture 01d: LangChain chains and MLflow tracing

**Status: Mandatory reading.**

In `lec_01c` you wrote `build_prompt`, `ask`, and `tag_question` by hand. Every LLM application has those same three parts, so libraries exist that name them and give them one common interface.
**LangChain** is the most widely used one, and lecture 2 builds the retrieval system on it.

Then the harder problem: once calls are chained together, what was *actually* sent to the model? Printing does not scale.
**MLflow tracing** records every step, and you already know MLflow from lecture 13.

**What this notebook covers**

- LangChain's parts: chat model, prompt template, output parser, and the `|` that chains them.
- `invoke` for one input, `batch` for many, structured output inside a chain.
- **Traces** and **spans**: what MLflow records, and how to read the prompt that was really sent.

</div>

In [1]:
import mlflow
import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

from llm_course.checks import check_mlflow

In [2]:
# --- Course configuration: edit TIER only -------------------------------------
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

import os
from pathlib import Path

TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
EXPERIMENT = "llm-course-01-basics"
REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. Why a framework

Your `ask` function from `lec_01c` took a prompt, called Ollama, and returned text. That is a **chat model** in LangChain:
an object with an `invoke` method. The other parts have names too:

| LangChain name | What you wrote in `lec_01c` | Job |
| --- | --- | --- |
| **chat model** (`ChatOllama`) | `client.chat(...)` | sends messages to a model, returns a message |
| **prompt template** (`ChatPromptTemplate`) | `build_prompt(question, style)` | fills named slots into messages |
| **output parser** (`StrOutputParser`) | `.message.content.strip()` | turns the model's message into what your code wants |
| **chain** (built with `\|`) | calling them in order | one object with `invoke`, `batch`, `stream` |

The value is the *common interface*: a chain built on Ollama today runs on a hosted model tomorrow by changing one line,
and lecture 2's retriever plugs in the same way. The cost is one more library whose names you must learn, and which changes between versions;
this course pins LangChain 1.x in `uv.lock`.

</div>

In [3]:
llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,  # the same switch as think=False in the ollama client
    temperature=0,
    num_predict=200,
)

reply = llm.invoke("In one sentence, what is a DataFrame?")
print(reply.content)

A DataFrame is a two-dimensional data structure used in Python's pandas library to store and manipulate tabular data, such as rows and columns, with support for various data types and operations.


In [4]:
reply.usage_metadata  # tokens in and out, the same numbers lec_01a computed by hand

{'input_tokens': 25, 'output_tokens': 38, 'total_tokens': 63}

<div style="text-align: justify; max-width: 90ch">

`invoke` returns an `AIMessage`: the `assistant` role from `lec_01c`, as an object with `content` and metadata.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Prompt templates

A template holds the messages with **named slots** in curly braces. `invoke` with a dict fills the slots and returns the rendered messages,
without calling any model, so you can look at exactly what will be sent.

</div>

In [5]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a teaching assistant for a Python course. Answer in at most {sentences} sentences.",
        ),
        ("human", "{question}"),
    ]
)

rendered = prompt.invoke(
    {"sentences": 2, "question": "What does the axis argument mean in pandas drop()?"}
)
for message in rendered.messages:
    print(f"[{message.type}] {message.content}")

[system] You are a teaching assistant for a Python course. Answer in at most 2 sentences.
[human] What does the axis argument mean in pandas drop()?


<div style="text-align: justify; max-width: 90ch">

Curly braces are the slot syntax, which bites the first time a template contains literal braces, for example a JSON example for the model.

</div>

In [6]:
# WRONG: the JSON example's braces are read as slots named '"category"'
broken = ChatPromptTemplate.from_template(
    'Reply as JSON like {"category": "pandas"}. Question: {question}'
)
try:
    broken.invoke({"question": "How do I merge tables?"})
except KeyError as error:
    print("KeyError:", error)

KeyError: 'Input to ChatPromptTemplate is missing variables {\'"category"\'}.  Expected: [\'"category"\', \'question\'] Received: [\'question\']\nNote: if you intended {"category"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"category"}}\'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT '


In [7]:
# RIGHT: double the braces you want to keep
fixed = ChatPromptTemplate.from_template(
    'Reply as JSON like {{"category": "pandas"}}. Question: {question}'
)
print(fixed.invoke({"question": "How do I merge tables?"}).messages[0].content)

Reply as JSON like {"category": "pandas"}. Question: How do I merge tables?


<div style="text-align: justify; max-width: 90ch">

## 3. Chains

`|` joins parts into a **chain**: the output of each becomes the input of the next. `prompt | llm | StrOutputParser()` reads as "fill the template, send it, keep the text".

</div>

In [8]:
chain = prompt | llm | StrOutputParser()

print(
    chain.invoke(
        {
            "sentences": 2,
            "question": "Why do we scale features after the train/test split?",
        }
    )
)

We scale features after the train/test split to ensure that features of different scales do not disproportionately influence the model. This helps improve the performance of algorithms that are sensitive to scale, such as those using Euclidean distances.


<div style="text-align: justify; max-width: 90ch">

`batch` runs the chain over many inputs. Ollama serves one request at a time on a laptop, so we cap concurrency at 1; a hosted model would take 10 in parallel with the same line.

</div>

In [9]:
questions = [
    "What is the difference between loc and iloc?",
    "When is a silhouette score useful?",
    "What does random_state do?",
]
inputs = [{"sentences": 1, "question": q} for q in questions]

answers = chain.batch(inputs, config={"max_concurrency": 1})
pd.DataFrame({"question": questions, "answer": answers})

,question,answer
0,What is the difference between loc and iloc?,"`loc` is used for label-based indexing (e.g., ..."
1,When is a silhouette score useful?,A silhouette score is useful for evaluating th...
2,What does random_state do?,The `random_state` parameter ensures reproduci...


<div style="text-align: justify; max-width: 90ch">

Structured output from `lec_01c` is a method on the chat model: `with_structured_output` takes the same JSON schema and returns a model that answers with a dict.
It slots into a chain like any other part.

</div>

In [10]:
TAG_SCHEMA = {
    "title": "ForumTag",
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["pandas", "plots", "sklearn", "python-basics", "other"],
        },
        "needs_code": {"type": "boolean"},
    },
    "required": ["category", "needs_code"],
}

tag_prompt = ChatPromptTemplate.from_template(
    "Tag this Python-course forum question.\n\nQuestion: {question}"
)
tag_chain = tag_prompt | llm.with_structured_output(TAG_SCHEMA, method="json_schema")

tag_chain.invoke({"question": "How do I change the colours in a seaborn heatmap?"})

{'category': 'python-basics', 'needs_code': True}

<div style="text-align: justify; max-width: 90ch">

## 4. Seeing inside: MLflow tracing

A chain hides its steps. When an answer is wrong, was it the template, the retrieved text (lecture 2), or the model? In lecture 13 MLflow recorded *runs*:
parameters and metrics of a training job. For LLM applications it records **traces**:

| Term | Meaning |
| --- | --- |
| **trace** | one end-to-end call of your application, for example one `chain.invoke` |
| **span** | one step inside it (template, model call, parser), with its input, output and duration; spans nest |
| **autolog** | one line that makes LangChain report every step to MLflow, no code changes |

The tracking server from guide 00e must be running; the check below says so if it is not.

</div>

In [11]:
check_mlflow(MLFLOW_URI)

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()

MLflow OK at http://127.0.0.1:5010


In [12]:
answer = chain.invoke({"sentences": 2, "question": "What is data leakage?"})
print(answer)

mlflow.flush_trace_async_logging()  # traces are sent in the background; wait for this one
trace = mlflow.get_trace(mlflow.get_last_active_trace_id())

pd.DataFrame(
    [
        {
            "span": s.name,
            "type": str(s.span_type).split(".")[-1],
            "seconds": round((s.end_time_ns - s.start_time_ns) / 1e9, 2),
        }
        for s in trace.data.spans
    ]
)

Data leakage occurs when a model is trained on data that includes information from the target variable, leading to overly optimistic performance estimates.


,span,type,seconds
0,RunnableSequence,CHAIN,0.36
1,ChatPromptTemplate,CHAIN,0.00
2,ChatOllama,CHAT_MODEL,0.36
3,StrOutputParser,CHAIN,0.00


<div style="text-align: justify; max-width: 90ch">

Four spans, one per part of the chain, and the model call took all the time. Open the experiment in the MLflow UI, **Traces** tab, and click the newest trace.

### ⏸ Pause point

Click the `ChatOllama` span and read its **inputs**. That is the prompt the model *actually* received: the system message with `{sentences}` filled in, then the question.
Nobody typed that text; the template produced it. The same view, read programmatically:

</div>

In [13]:
model_span = next(s for s in trace.data.spans if s.name == "ChatOllama")

for message in model_span.inputs["messages"]:
    print(f"[{message['role']}] {message['content']}")
print()
print("tokens:", trace.info.token_usage, "| total ms:", trace.info.execution_duration)

[system] You are a teaching assistant for a Python course. Answer in at most 2 sentences.
[user] What is data leakage?

tokens: {'input_tokens': 44, 'output_tokens': 26, 'total_tokens': 70} | total ms: 361


<div style="text-align: justify; max-width: 90ch">

This is the habit that carries through the course: when an answer is wrong, **read the trace before touching the code**.
In lecture 2 the trace will show which documents the retriever handed to the model; in lecture 3 it feeds the evaluation.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. Swapping the model is one line

Everything above used `ChatOllama`. A hosted model, for a job where privacy allows it and latency matters, is the same chain with a different first line:

```python
from langchain_openai import ChatOpenAI          # any OpenAI-compatible provider
llm = ChatOpenAI(model="...", base_url="...", api_key=os.environ["PROVIDER_KEY"])
chain = prompt | llm | StrOutputParser()        # unchanged
```

`mlflow.langchain.autolog()` traces it just the same. The optional `lec_01f` tries it against a hosted endpoint.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- LangChain names the parts you already wrote: **chat model**, **prompt template**, **output parser**; `|` chains them into one object with `invoke`, `batch`, `stream`.
- Templates render without calling the model; literal braces must be doubled.
- `with_structured_output` puts a JSON schema inside a chain.
- **MLflow tracing**: `mlflow.langchain.autolog()` records a **trace** of **spans** per call; the model span shows the prompt actually sent and the tokens used.
- Read the trace before changing the code.

**Next: lecture 2**, where the model stops answering from memory and starts answering from *your* documents: embeddings, retrieval, and a hand-built RAG.

</div>